In [2]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TensorFlow: 2.10.1
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow import keras
from tensorflow.keras import layers


# 1. Wczytanie danych
csv_path = r"C:\Users\tomas\Desktop\github\upp_csde\metody_si\projekt\dane\merged_data.csv"
df = pd.read_csv(csv_path, encoding="utf-8")

# Check dataframe
if df.empty:
    raise ValueError(f"DataFrame loaded from {csv_path} is empty. Check the file path and contents.")

# 2. Data
df["DATA"] = pd.to_datetime(dict(year=df["ROK"], month=df["MC"], day=df["DZ"]))

# 3. Sortowanie po czasie
df = df.sort_values("DATA").reset_index(drop=True)

# 4. Target = średnia temperatura z następnego dnia
df["TARGET"] = df["STD"].shift(-1)

# 5. Proste cechy opóźnione
df["STD_lag1"] = df["STD"].shift(1)
df["TMAX_lag1"] = df["TMAX"].shift(1)
df["TMIN_lag1"] = df["TMIN"].shift(1)

# 6. Usunięcie braków - ONLY in columns we need for the model
df = df.dropna(subset=["STD_lag1", "TMAX_lag1", "TMIN_lag1", "TARGET"])
if df.empty:
    raise ValueError("No data left after creating lags / dropping NA. Check input data.")

# 7. X i y
X = df[["STD_lag1", "TMAX_lag1", "TMIN_lag1"]].values
y = df["TARGET"].values

# 8. Podział czasowy
split = int(len(df) * 0.8)
# Ensure at least one sample in train and test if possible
if split < 1:
    split = 1 if len(df) > 1 else len(df)  # if only 1 sample, go to later check

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]


In [ ]:
# Check sizes
if X_train.shape[0] == 0:
    raise ValueError("Training set is empty after split. Increase data or adjust split ratio.")
if X_test.shape[0] == 0:
    # If no test set, keep some training for testing
    if X_train.shape[0] > 1:
        X_test = X_train[-1:].copy()
        y_test = y_train[-1:].copy()
        X_train = X_train[:-1]
        y_train = y_train[:-1]
    else:
        raise ValueError("Not enough data for testing. Need at least 2 samples.")

# 9. Skalowanie (guard for zero std)
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std_fixed = np.where(std == 0, 1.0, std)
X_train = (X_train - mean) / std_fixed
X_test = (X_test - mean) / std_fixed

# 10. Model
model = keras.Sequential([
    layers.Input(shape=(3,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(1)
])

# 11. Kompilacja
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

# 12. Early stopping
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=30,
    restore_best_weights=True
)

# 13. Trenowanie
# avoid validation_split when training set is too small
val_split = 0.2 if X_train.shape[0] >= 10 else 0.0
fit_kwargs = dict(
    epochs=200,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)
if val_split > 0:
    fit_kwargs.update(dict(validation_split=val_split))
else:
    # use test set as validation if available
    fit_kwargs.update(dict(validation_data=(X_test, y_test)))

history = model.fit(
    X_train,
    y_train,
    **fit_kwargs
)

# 14. Predykcja
y_pred = model.predict(X_test).flatten()

# 15. Ocena
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print("MAE:", round(mae, 3))
print("RMSE:", round(rmse, 3))

# 16. Wyniki
wyniki = pd.DataFrame({
    "data": df["DATA"].iloc[split:split + len(y_test)].values,
    "rzeczywiste": y_test,
    "przewidywane": y_pred
})

print(wyniki.head(20))

Epoch 1/200
2657/2657 [==============================] - 6s 2ms/step - loss: 15.9237 - mae: 2.6778 - val_loss: 7.9407 - val_mae: 2.0891
Epoch 2/200
2657/2657 [==============================] - 6s 2ms/step - loss: 7.9449 - mae: 2.0948 - val_loss: 7.8656 - val_mae: 2.0732
Epoch 3/200
2657/2657 [==============================] - 5s 2ms/step - loss: 7.8999 - mae: 2.0861 - val_loss: 7.8863 - val_mae: 2.0752
Epoch 4/200
2657/2657 [==============================] - 5s 2ms/step - loss: 7.8875 - mae: 2.0836 - val_loss: 7.9271 - val_mae: 2.0842
Epoch 5/200
2657/2657 [==============================] - 5s 2ms/step - loss: 7.8820 - mae: 2.0827 - val_loss: 7.9139 - val_mae: 2.0796
Epoch 6/200
2657/2657 [==============================] - 6s 2ms/step - loss: 7.8793 - mae: 2.0824 - val_loss: 7.8733 - val_mae: 2.0740
Epoch 7/200
2657/2657 [==============================] - 5s 2ms/step - loss: 7.8749 - mae: 2.0817 - val_loss: 7.8718 - val_mae: 2.0748
Epoch 8/200
2657/2657 [==============================]

In [10]:
print(len(history.history["loss"]))

118


In [11]:
# Save the entire model (architecture + weights + training config)
model.save("temperature_model.keras")